# Decision Tree (Stroke Prediction):
- This decision tree algorithm was trained to discover/classify the "stroke" column from the healthcare .csv file;
- The data was split into 80% for training and 20% for testing;

In [161]:
import os 


graphviz_bin = r"C:\Program Files\Graphviz\bin"
if graphviz_bin not in os.environ["PATH"]:
    os.environ["PATH"] += os.pathsep + graphviz_bin

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split as tts
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder 
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.tree import export_graphviz
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import graphviz

## Load and filter the data:
- For this filtering, I am using a different method. I planned a study with Claude and am using its filtering method. 

In [162]:
data_base = pd.read_csv('./healthcare-dataset-stroke-data.csv')

SEED = 12

SCORING = 'average_precision'

y = data_base['stroke']
x = data_base.drop(columns=['stroke', 'id'])

x_train, x_test, y_train, y_test = tts(x, y, test_size = 0.2, random_state = SEED, stratify = y)

base = DummyClassifier(strategy = 'most_frequent').fit(x_train, y_train)

base

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](2,)","[0.95,0.05]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](10,)","['gender','age','hypertension',...,'avg_glucose_level','bmi', 'smoking_status']"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,2
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,10
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


Analyze the baseline accuracy:

In [163]:
base_acurracy = round(base.score(x_test, y_test), 4)

print(f'Base Accuracy: {base_acurracy}')

Base Accuracy: 0.9511


### Building pipelines:

In [164]:
col_nums = x_train.select_dtypes(include = 'number').columns
col_cats = x_train.select_dtypes(exclude = 'number').columns

In [165]:
def build_pipeline(with_indicator):
    prep = ColumnTransformer([
        ('num', SimpleImputer(strategy = 'median', add_indicator = with_indicator), col_nums),
        ('cat', OneHotEncoder(handle_unknown = 'ignore'), col_cats)
    ], verbose_feature_names_out = False)

    return Pipeline([
        ('prep', prep),
        ('tree', DecisionTreeClassifier(random_state = SEED, class_weight = 'balanced'))
    ])

### Hyperparameter grid and cross-validation:

In [166]:
param_grid = {
    'tree__max_depth': [2, 3, 4, 5, 6, 8, 10],
    'tree__min_samples_leaf': [1, 2, 5, 10, 20, 50]
}

cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = SEED)

### Training with and without the missing BMI indicator:
#### Recall:
With Indicator: False
Best params: {'tree__max_depth': 2, 'tree__min_samples_leaf': 5}
recall (CV mean): 0.8746

With Indicator: True
Best params: {'tree__max_depth': 2, 'tree__min_samples_leaf': 5}
recall (CV mean): 0.8746

#### f1:
With Indicator: False
Best params: {'tree__max_depth': 6, 'tree__min_samples_leaf': 5}
f1 (CV mean): 0.2303

With Indicator: True
Best params: {'tree__max_depth': 4, 'tree__min_samples_leaf': 5}
f1 (CV mean): 0.2415

#### Avarage Precision:
With Indicator: False
Best params: {'tree__max_depth': 6, 'tree__min_samples_leaf': 20}
average_precision (CV mean): 0.1587

With Indicator: True
Best params: {'tree__max_depth': 6, 'tree__min_samples_leaf': 5}
average_precision (CV mean): 0.1914

In [167]:
results = {}

for with_indicator in [False, True]:
    search = GridSearchCV(build_pipeline(with_indicator), param_grid, cv = cv, scoring = SCORING)
    search.fit(x_train, y_train)
    results[with_indicator] = search

    print(f'With Indicator: {with_indicator}')
    print(f'Best params: {search.best_params_}')
    print(f'{SCORING} (CV mean): {search.best_score_:.4f}\n')



With Indicator: False
Best params: {'tree__max_depth': 6, 'tree__min_samples_leaf': 20}
average_precision (CV mean): 0.1587

With Indicator: True
Best params: {'tree__max_depth': 6, 'tree__min_samples_leaf': 5}
average_precision (CV mean): 0.1914



### Choosing the model through cross-validation:

In [168]:
use_indicator = results[True].best_score_ > results[False].best_score_

model = results[use_indicator].best_estimator_

print(f'Using indicator: {use_indicator}')

Using indicator: True


### Create the tree graph:

In [169]:
tree_model = model['tree']

feature_name = model['prep'].get_feature_names_out()

dot_data = export_graphviz(
    tree_model,
    out_file = None,
    feature_names = feature_name,
    class_names = ['No Stroke', 'Stroke'],
    rounded = True,
    filled = True,
    max_depth = 3
)

graph = graphviz.Source(dot_data)
graph.render('Stroke Decision Tree', format = 'png', cleanup = True)
graph.view()

'Stroke Decision Tree.pdf'

### Prediction with the remaining 20% of the data:

In [170]:
predictions = model.predict(x_test)

predictions

array([0, 1, 0, ..., 1, 0, 0], shape=(1022,))

### Metric analysis:

In [171]:
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')

Accuracy: 0.7769
Precision: 0.1382
Recall: 0.6800
F1 Score: 0.2297


In [172]:
report = classification_report(y_test, predictions, target_names = ['No Stroke', 'Stroke'])

print(report)

              precision    recall  f1-score   support

   No Stroke       0.98      0.78      0.87       972
      Stroke       0.14      0.68      0.23        50

    accuracy                           0.78      1022
   macro avg       0.56      0.73      0.55      1022
weighted avg       0.94      0.78      0.84      1022



### Finding incorrect predictions:

In [173]:
errors = x_test[y_test != predictions].copy()

errors['actual'] = y_test[y_test != predictions]
errors['predicted'] = predictions[y_test != predictions]

errors.head()

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,actual,predicted
4513,Female,62.0,0,0,Yes,Private,Rural,117.63,27.1,formerly smoked,0,1
1752,Female,76.0,1,1,Yes,Private,Rural,102.08,31.0,smokes,0,1
946,Female,47.0,0,0,Yes,Private,Rural,204.63,43.4,never smoked,0,1
85,Male,54.0,0,0,Yes,Private,Rural,96.97,29.1,smokes,1,0
1852,Male,78.0,0,0,Yes,Private,Rural,100.09,30.5,Unknown,0,1


### Confusion matrix:

In [174]:
matrix = confusion_matrix(y_test, predictions)

matrix_data = pd.DataFrame(matrix, index = ['No Stroke', 'Stroke'], columns = ['Predicted No Stroke', 'Predicted Stroke'])

matrix_data

,Predicted No Stroke,Predicted Stroke
No Stroke,760,212
Stroke,16,34


### What the tree used to make its decisions:

In [175]:
importances = pd.Series(tree_model.feature_importances_, index= feature_name).sort_values(ascending = False)

importances

age                               0.710373
avg_glucose_level                 0.117041
bmi                               0.077000
missingindicator_bmi              0.041568
hypertension                      0.030247
smoking_status_never smoked       0.018419
smoking_status_smokes             0.004945
smoking_status_Unknown            0.000239
work_type_Private                 0.000168
heart_disease                     0.000000
ever_married_No                   0.000000
gender_Female                     0.000000
gender_Other                      0.000000
gender_Male                       0.000000
work_type_Never_worked            0.000000
work_type_Govt_job                0.000000
ever_married_Yes                  0.000000
work_type_Self-employed           0.000000
Residence_type_Urban              0.000000
Residence_type_Rural              0.000000
work_type_children                0.000000
smoking_status_formerly smoked    0.000000
dtype: float64

Obs: Claude filtering methedo wasn't good as simple way i used on the other decision tree algorithm